# Atlas-scale Tissue VAE — Embedding Notebook

Run top-to-bottom with `E:\atlas_tissue_representation_fork\.venv\Scripts\python.exe`.

| Representation | Status | Dimension |
|---|---|---:|
| Sample embedding | Native VAE latent mean `mu` | 121 |
| Static gene embedding | N/A | N/A |
| Contextual gene embedding | N/A | N/A |

The model treats genes as numeric input features. It does not define a native per-gene embedding table or per-gene contextual hidden states.


In [1]:
from pathlib import Path
import sys, time
import joblib
import numpy as np
import pandas as pd
import torch
import sklearn

ROOT=Path(r"E:\atlas_tissue_representation_fork")
MODEL_PATH=ROOT/"model"/"vae_tissue.final_model.pth"
ARTIFACT_PATH=ROOT/"model"/"vae_tissue.artifacts.joblib"
INPUT_PATH=ROOT/"gex.lung_100.csv"
OUTPUT_DIR=ROOT/"outputs"/"atlas_tissue_vae"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

assert MODEL_PATH.exists(),MODEL_PATH
assert ARTIFACT_PATH.exists(),ARTIFACT_PATH
assert INPUT_PATH.exists(),INPUT_PATH

DEVICE=torch.device("cpu")
print("Python:",sys.executable)
print("PyTorch:",torch.__version__)
print("scikit-learn:",sklearn.__version__)
print("Device:",DEVICE)

print("\nLoading full pretrained VAE...")
t0=time.perf_counter()
model=torch.load(MODEL_PATH,map_location=DEVICE,weights_only=False)
model.eval()
model_load_time=time.perf_counter()-t0
print("Model load seconds:",round(model_load_time,2))

artifacts=joblib.load(ARTIFACT_PATH)
gene_list=list(artifacts["feature_lists"]["gex"])
scaler=artifacts["transforms"]["gex"]
print("Model genes:",len(gene_list))

df=pd.read_csv(INPUT_PATH,index_col=0)
gcols=len(set(df.columns)&set(gene_list))
grows=len(set(df.index)&set(gene_list))
if grows>gcols:
    df=df.T
    print("Input transposed to samples x genes.")

aligned=pd.DataFrame(0.0,index=df.index,columns=gene_list)
common_genes=[g for g in gene_list if g in df.columns]
aligned.loc[:,common_genes]=df.loc[:,common_genes].values
aligned=aligned.fillna(0)
gene_coverage=100*len(common_genes)/len(gene_list)

# Pass DataFrame so feature names/order are preserved.
scaled=scaler.transform(aligned)
X=torch.tensor(scaled,dtype=torch.float32,device=DEVICE)

print("Input samples:",df.shape[0])
print("Original input genes:",df.shape[1])
print("Matched genes:",len(common_genes))
print("Missing genes zero-filled:",len(gene_list)-len(common_genes))
print("Gene coverage:",round(gene_coverage,2),"%")
print("Model input shape:",tuple(X.shape))
print("Setup complete.")


Python: e:\atlas_tissue_representation_fork\.venv\Scripts\python.exe
PyTorch: 2.14.1+cpu
scikit-learn: 1.9.1
Device: cpu

Loading full pretrained VAE...


e:\atlas_tissue_representation_fork\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Model load seconds: 13.07
Model genes: 16115


e:\atlas_tissue_representation_fork\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
e:\atlas_tissue_representation_fork\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


Input transposed to samples x genes.
Input samples: 100
Original input genes: 37298
Matched genes: 16069
Missing genes zero-filled: 46
Gene coverage: 99.71 %
Model input shape: (100, 16115)
Setup complete.


In [2]:
# CELL 1 — NATIVE SAMPLE EMBEDDING
# Repository path: h=model.encoders[0](X); mu=model.FC_mean(h[0])
t0=time.perf_counter()
with torch.no_grad():
    h=model.encoders[0](X)
    mu=model.FC_mean(h[0])
elapsed=time.perf_counter()-t0

sample_embeddings=mu.detach().cpu().numpy().astype(np.float32)
sample_names=np.asarray(df.index,dtype=object)

assert sample_embeddings.shape==(df.shape[0],121)
assert np.isnan(sample_embeddings).sum()==0
assert np.isinf(sample_embeddings).sum()==0

np.save(OUTPUT_DIR/"sample_embeddings.npy",sample_embeddings)
np.save(OUTPUT_DIR/"sample_names.npy",sample_names)
pd.DataFrame(
    sample_embeddings,index=df.index,
    columns=[f"z{i}" for i in range(sample_embeddings.shape[1])]
).to_csv(OUTPUT_DIR/"sample_embeddings.csv")

print("ATLAS TISSUE VAE SAMPLE EMBEDDING")
print("="*60)
print("Samples:",sample_embeddings.shape[0])
print("Original input genes:",df.shape[1])
print("Model genes:",len(gene_list))
print("Matched genes:",len(common_genes))
print("Gene coverage:",round(gene_coverage,2),"%")
print("Model input shape:",tuple(X.shape))
print("Output shape:",sample_embeddings.shape)
print("Dimension:",sample_embeddings.shape[-1])
print("dtype:",sample_embeddings.dtype)
print("min:",sample_embeddings.min())
print("max:",sample_embeddings.max())
print("mean:",sample_embeddings.mean())
print("std:",sample_embeddings.std())
print("NaN:",np.isnan(sample_embeddings).sum())
print("Inf:",np.isinf(sample_embeddings).sum())
print("Inference time seconds:",round(elapsed,6))
print("Seconds per sample:",round(elapsed/sample_embeddings.shape[0],6))
print("Model load seconds:",round(model_load_time,2))
print("Device:",DEVICE)
print("Saved:",OUTPUT_DIR/"sample_embeddings.npy")


ATLAS TISSUE VAE SAMPLE EMBEDDING
Samples: 100
Original input genes: 37298
Model genes: 16115
Matched genes: 16069
Gene coverage: 99.71 %
Model input shape: (100, 16115)
Output shape: (100, 121)
Dimension: 121
dtype: float32
min: -2839.358
max: 3851.8972
mean: -2.0864334
std: 228.48137
NaN: 0
Inf: 0
Inference time seconds: 0.014483
Seconds per sample: 0.000145
Model load seconds: 13.07
Device: cpu
Saved: E:\atlas_tissue_representation_fork\outputs\atlas_tissue_vae\sample_embeddings.npy


## Static gene embedding

N/A for this architecture. Genes are coordinates in a fixed numerical expression vector, rather than learned gene tokens. No synthetic gene representation is fabricated for this benchmark.


In [3]:
# CELL 2 — STATIC GENE EMBEDDING
# N/A: genes are fixed numeric input features; there is no native per-gene embedding table.
status={
    "supported":False,
    "dimension":None,
    "reason":"Genes are fixed numeric input features; the VAE exposes no native learned per-gene embedding."
}
print("STATIC GENE EMBEDDING")
print("="*60)
print("Supported:",status["supported"])
print("Dimension:",status["dimension"])
print("Reason:",status["reason"])


STATIC GENE EMBEDDING
Supported: False
Dimension: None
Reason: Genes are fixed numeric input features; the VAE exposes no native learned per-gene embedding.


## Contextual gene embedding

N/A for this architecture. The VAE compresses the full 16,115-feature expression vector into a sample-level latent representation and does not expose a `sample × gene × hidden_dimension` tensor.


In [4]:
# CELL 3 — CONTEXTUAL GENE EMBEDDING
# N/A: the encoder returns a sample-level latent vector, not per-gene contextual states.
status={
    "supported":False,
    "dimension":None,
    "reason":"The VAE encoder produces a sample-level latent vector and exposes no per-gene contextual hidden states."
}
print("CONTEXTUAL GENE EMBEDDING")
print("="*60)
print("Supported:",status["supported"])
print("Dimension:",status["dimension"])
print("Reason:",status["reason"])


CONTEXTUAL GENE EMBEDDING
Supported: False
Dimension: None
Reason: The VAE encoder produces a sample-level latent vector and exposes no per-gene contextual hidden states.


## Representation summary

| Representation | Output | Dimension | Native? |
|---|---|---:|---|
| Sample embedding | `N × 121` | 121 | Yes |
| Static gene embedding | N/A | N/A | No |
| Contextual gene embedding | N/A | N/A | No |

For `gex.lung_100.csv`, the validated native output is `(100, 121)`.

Native extraction:

```python
h = model.encoders[0](X)
mu = model.FC_mean(h[0])
```
